# Model comparison
Compare multinomial vs. XGBoost vs. Poisson outcomes on the validation slice.

In [ ]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from src.data.data_loader import DataLoader
from src.data.data_splitter import temporal_split
from src.features.build_features import build_match_feature_matrix, select_feature_columns
from src.ratings.rating_ensemble import RatingEnsemble
from src.models.multinomial_model import MultinomialOutcomeModel
from src.models.xgboost_model import XGBoostOutcomeModel
from src.models.poisson_model import PoissonScoreModel
from src.training.evaluator import Evaluator

In [ ]:
matches = DataLoader().load_matches()
composite = RatingEnsemble().fit(matches).composite_table()
matrix = build_match_feature_matrix(matches, pd.DataFrame(), composite).dropna(subset=['outcome'])
split = temporal_split(matrix, validation_year=2024)
cols = select_feature_columns(matrix)
print('train rows', len(split.train), 'val rows', len(split.validation))

In [ ]:
evaluator = Evaluator()
mn = MultinomialOutcomeModel(feature_columns=cols).fit(split.train[cols], split.train['outcome'])
xgb = XGBoostOutcomeModel(feature_columns=cols).fit(split.train[cols], split.train['outcome'])
poisson = PoissonScoreModel().fit(split.train, split.train['outcome'])
if split.validation.empty:
    print('Validation slice empty; skipping evaluation')
else:
    for name, model in [('multinomial', mn), ('xgboost', xgb)]:
        proba = model.predict_proba(split.validation[cols])
        print(name, evaluator.evaluate(split.validation['outcome'].tolist(), proba).to_dict())